# Milestone 7: workspace reuse and linear drag
This notebook inspects the reversible circuit, not a real QPU run. The underlying model is finite-volume shallow water, not D2Q9 LBM.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "milestone7": root = root.parent
sys.path.insert(0, str(root))
from milestone7.optimization import CompactFlux
from milestone7.drag import DragOracle
from milestone7.solver import Damped2D
import numpy as np


## 1. Compare the same arithmetic function before and after workspace compaction

In [ ]:
a = CompactFlux(32, 1)
print("Work qubits:", a.original_ancillas, "->", a.ancillas)
print("Logical gates:", a.original_gates, "->", len(a.gates))
print("Flux ticks:", a.unpack(a.evaluate(a.pack(48, 3, -1))))


## 2. Reversible drag: retained input and damped output

In [ ]:
d = DragOracle(8)
q = 32
for step in range(10):
    print(step, q, q/32, (7/8)**step)
    q = d.evaluate(q)


## 3. Small grid gate simulation and reversal

In [ ]:
c = Damped2D(np.zeros((1, 4), dtype=bool), 16)
s = np.zeros((1, 4, 3), dtype=int)
s[0, :2, 0] = 24
for step in range(4):
    print(step, s[0,:,0]/16, "mass ticks:", s[:,:,0].sum())
    s = c.step(s, verify_inverse=True)


## 4. Read archived full experiment results
The 16×16 run used a basis gate simulator with cached local gate evaluations. See README for SI scaling, limitations and reproduction.

In [ ]:
import json
from IPython.display import Image, display
report = json.loads((root/"milestone7/results/report.json").read_text())
print(json.dumps(report, indent=2))
display(Image(filename=str(root/"milestone7/results/benchmark.png")))
